# Session 13 — Standard 20M TinyStories baseline

This is the first of the three assignment runs. It trains a decoder-only Transformer from random initialization on 50 million TinyStories tokens. The GPT-2 tokenizer is reused, but no pretrained model weights are loaded. The script prints the final validation loss, training throughput, peak CUDA memory, hardware, and exact run configuration.

## Run in Colab

1. Choose **Runtime → Change runtime type → GPU**. A T4 is a suitable starting point.
2. Run the cells in order. The first run downloads the public dataset and tokenizer and caches token IDs on Google Drive.
3. The training loop saves resumable checkpoints every 500 steps. If Colab disconnects, reconnect, rerun setup, and rerun the training cell; it resumes from the latest checkpoint.
4. Save the final printed results and `baseline_results.json` for the report. Training throughput excludes validation and checkpoint-save time; peak memory is CUDA allocated/reserved memory.

The baseline uses batch size 16, sequence length 256, and approximately 20M model parameters. Change these only deliberately and record the actual values. The saved token cache is shared by all three runs.

In [ ]:
%pip -q install "datasets>=3,<5" "transformers>=4.46,<5"

from pathlib import Path
import subprocess

repo = Path("/content/ERA_Portfolio")
if not repo.exists():
    subprocess.run(
        ["git", "clone", "https://github.com/dhruv369/ERA_Portfolio.git", str(repo)],
        check=True,
    )
%cd /content/ERA_Portfolio/Assignments/S13

In [ ]:
import os
import torch
from google.colab import drive

if not torch.cuda.is_available():
    raise RuntimeError("Enable a GPU in Runtime → Change runtime type before training.")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"PyTorch: {torch.__version__}; CUDA: {torch.version.cuda}")

drive.mount("/content/drive")
os.environ["HF_HOME"] = "/content/drive/MyDrive/ERA_S13/hf_cache"
os.environ["S13_DATA_DIR"] = "/content/drive/MyDrive/ERA_S13/data"
os.environ["S13_RUNS_DIR"] = "/content/drive/MyDrive/ERA_S13/runs"
os.environ["S13_RUN_DIR"] = "/content/drive/MyDrive/ERA_S13/baseline"
print("Run data and checkpoints will persist in Google Drive.")

## First: validate the GPU and model

Run this short check first. It verifies the tokenizer download, exact parameter count, and model output shape without downloading TinyStories or starting training. If it passes, continue to the long baseline run.

In [ ]:
!python -u session13_baseline.py --smoke-only

## Then: train the baseline

This cell can run for several hours. Do not change the batch size after starting a run: the checkpoint records the configuration and will reject incompatible resumes. The script reports actual training tokens, validation loss, tokens/second, and CUDA peak memory.

In [ ]:
!python -u session13_baseline.py

## Run 2: reversible model at the same batch size

This model uses **exact additive reversible coupling**. Each block computes `y1 = x1 + F(x2)` and `y2 = x2 + G(y1)`; the backward pass reconstructs the inputs as `x2 = y2 - G(y1)` and `x1 = y1 - F(x2)`, then recomputes block activations. It is an exact reversible-coupling variant (not an Euler or midpoint ODE solver). The architecture has about 19.26M parameters and uses the same tokenizer, data, token budget, sequence length, optimizer, and batch size as the baseline.

Run this only after the baseline completes. It checkpoints to Drive and can resume if Colab disconnects.

In [ ]:
!python -u session13_reversible.py --batch-size 16 --run-name reversible-fixed

## Run 3: reversible model at the maximum fitting batch size

The script probes batches by running forward, backward, and an optimizer step, then uses a binary search to find the largest tested batch that fits. It reports CUDA out-of-memory as a failed probe, clears temporary allocations, and then trains the 50-million-token run at the largest successful batch. This run has its own checkpoint directory.

In [ ]:
!python -u session13_reversible.py --find-max-batch --batch-size 16 --run-name reversible-max

## Compare measurements

After all three runs finish, load their result JSON files from Drive and use the recorded validation loss, actual token count, throughput, peak CUDA memory, hardware, and configuration in the S13 README. Do not compare runs with different hardware or precision settings as if batch size or reversibility alone caused the difference.

In [ ]:
import json
from pathlib import Path

result_files = {
    "standard": Path("/content/drive/MyDrive/ERA_S13/baseline/baseline_results.json"),
    "reversible_fixed": Path("/content/drive/MyDrive/ERA_S13/runs/reversible-fixed/reversible_results.json"),
    "reversible_max_batch": Path("/content/drive/MyDrive/ERA_S13/runs/reversible-max/reversible_results.json"),
}
for name, path in result_files.items():
    if not path.exists():
        print(f"{name}: pending ({path})")
        continue
    result = json.loads(path.read_text())
    print(f"\n{name}:\n{json.dumps(result, indent=2)}")